# Notebook 04 — Semantic Harmonization

Loads the structural output and adds:
- `focus_raw` / `focus_normalized` — medical topic
- `primary_intent` — normalized intent taxonomy
- `dialogue_act` — turn-level dialogue act
- `medical_entities` — extracted medical entities
- `annotation_source` / `annotation_confidence` — provenance and reliability

All inference is rule-based and deterministic. Inferred labels are not treated as ground truth.

**Datasets:** MedQuAD (focus from `focus_area` label) and MedDialog (focus inferred from utterance text).

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
tqdm.pandas()

STRUCTURAL_DIR = Path('../data/processed/structural')
SEMANTIC_DIR   = Path('../data/processed/semantic')
SEMANTIC_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(STRUCTURAL_DIR / 'harmonized_structural.parquet')
print('Loaded structural dataset:', df.shape)
print('Datasets present:', df['source_dataset'].value_counts().to_dict())
display(df.head(4))

Loaded structural dataset: (257154, 8)
Datasets present: {'MedDialog': 224330, 'MedQuAD': 32824}


,dialogue_id,turn_id,speaker,utterance,source_dataset,original_id,source_label_raw,dialogue_origin
0,medquad_000000,0,user,What is (are) Glaucoma ?,MedQuAD,0,Glaucoma,constructed
1,medquad_000000,1,assistant,Glaucoma is a group of diseases that can damag...,MedQuAD,0,Glaucoma,constructed
2,medquad_000001,0,user,What causes Glaucoma ?,MedQuAD,1,Glaucoma,constructed
3,medquad_000001,1,assistant,"Nearly 2.7 million people have glaucoma, a lea...",MedQuAD,1,Glaucoma,constructed


## 1. Medical Focus

**MedQuAD:** `source_label_raw` = `focus_area` → used directly (`annotation_source = source_metadata`, confidence = 1.0)

**MedDialog:** No topic label in source → infer from utterance text via keyword patterns (`annotation_source = rule_based`, confidence = 0.6)

In [2]:
# ── Normalization dictionary ── inspect/modify here ──────────────────────────
FOCUS_NORMALIZATION = {
    'diabetes': 'diabetes', 'diabetes mellitus': 'diabetes', 'diabetic': 'diabetes',
    'type 1 diabetes': 'diabetes_type1', 'type 2 diabetes': 'diabetes_type2',
    'heart attack': 'myocardial_infarction', 'myocardial infarction': 'myocardial_infarction',
    'mi': 'myocardial_infarction', 'heart disease': 'heart_disease',
    'cardiovascular disease': 'heart_disease', 'coronary artery disease': 'coronary_artery_disease',
    'hypertension': 'hypertension', 'high blood pressure': 'hypertension',
    'depression': 'depression', 'major depressive disorder': 'depression',
    'anxiety': 'anxiety', 'anxiety disorder': 'anxiety', 'mental health': 'mental_health',
    'cancer': 'cancer', 'tumor': 'cancer', 'tumour': 'cancer', 'neoplasm': 'cancer',
    'breast cancer': 'breast_cancer', 'lung cancer': 'lung_cancer',
    'prostate cancer': 'prostate_cancer', 'colorectal cancer': 'colorectal_cancer',
    'asthma': 'asthma', 'copd': 'copd',
    'chronic obstructive pulmonary disease': 'copd',
    'migraine': 'migraine', 'headache': 'headache',
    "alzheimer's disease": 'alzheimers_disease', 'alzheimers disease': 'alzheimers_disease',
    "parkinson's disease": 'parkinsons_disease', 'parkinsons disease': 'parkinsons_disease',
    'stroke': 'stroke', 'arthritis': 'arthritis',
    'rheumatoid arthritis': 'rheumatoid_arthritis', 'osteoarthritis': 'osteoarthritis',
    'back pain': 'back_pain', 'covid': 'covid_19', 'covid-19': 'covid_19',
    'coronavirus': 'covid_19', 'influenza': 'influenza', 'flu': 'influenza',
    'pneumonia': 'pneumonia', 'ibs': 'ibs', 'irritable bowel syndrome': 'ibs',
    'gerd': 'gerd', 'acid reflux': 'gerd', 'gastroesophageal reflux': 'gerd',
    'thyroid': 'thyroid_disorder', 'hypothyroidism': 'hypothyroidism',
    'hyperthyroidism': 'hyperthyroidism', 'glaucoma': 'glaucoma',
    'macular degeneration': 'macular_degeneration',
}

def normalize_focus(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    key = str(raw).lower().strip()
    if key in FOCUS_NORMALIZATION:
        return FOCUS_NORMALIZATION[key]
    for k, v in FOCUS_NORMALIZATION.items():
        if k in key:
            return v
    return re.sub(r'[^a-z0-9]+', '_', key).strip('_') or None

# ── Keyword patterns for MedDialog focus inference ───────────────────────────
FOCUS_KEYWORD_PATTERNS = [
    ('diabetes',              re.compile(r'\bdiabet\w*\b', re.I)),
    ('hypertension',          re.compile(r'\b(hypertension|high blood pressure)\b', re.I)),
    ('myocardial_infarction', re.compile(r'\b(heart attack|myocardial infarction)\b', re.I)),
    ('heart_disease',         re.compile(r'\b(heart disease|cardiovascular|cardiac)\b', re.I)),
    ('depression',            re.compile(r'\b(depress\w+|major depressive)\b', re.I)),
    ('anxiety',               re.compile(r'\banxiet\w+\b', re.I)),
    ('asthma',                re.compile(r'\basthma\b', re.I)),
    ('copd',                  re.compile(r'\b(copd|chronic obstructive)\b', re.I)),
    ('cancer',                re.compile(r'\b(cancer|tumor|tumour|neoplasm|carcinoma)\b', re.I)),
    ('back_pain',             re.compile(r'\b(back pain|lumbar|spinal)\b', re.I)),
    ('migraine',              re.compile(r'\bmigraine\b', re.I)),
    ('headache',              re.compile(r'\bheadache\b', re.I)),
    ('stroke',                re.compile(r'\bstroke\b', re.I)),
    ('arthritis',             re.compile(r'\barthritis\b', re.I)),
    ('covid_19',              re.compile(r'\b(covid|coronavirus|sars-cov)\b', re.I)),
    ('influenza',             re.compile(r'\b(influenza|\bflu\b)\b', re.I)),
    ('pneumonia',             re.compile(r'\bpneumonia\b', re.I)),
    ('ibs',                   re.compile(r'\b(ibs|irritable bowel)\b', re.I)),
    ('gerd',                  re.compile(r'\b(gerd|acid reflux|heartburn|gastroesophageal)\b', re.I)),
    ('thyroid_disorder',      re.compile(r'\bthyroid\b', re.I)),
    ('kidney_disease',        re.compile(r'\b(kidney|renal)\b', re.I)),
    ('liver_disease',         re.compile(r'\b(liver|hepatic|hepatitis)\b', re.I)),
    ('skin_condition',        re.compile(r'\b(skin|rash|eczema|psoriasis|dermat\w+)\b', re.I)),
    ('infection',             re.compile(r'\b(infection|bacterial|viral|antibiotic)\b', re.I)),
    ('vertigo',               re.compile(r'\b(vertigo|dizziness|dizzy)\b', re.I)),
    ('pregnancy',             re.compile(r'\b(pregnan|prenatal|maternal|obstetric)\b', re.I)),
    ('pain',                  re.compile(r'\b(pain|ache|sore|hurt\w*)\b', re.I)),
    ('fever',                 re.compile(r'\b(fever|temperature)\b', re.I)),
]

def infer_focus_from_text(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return None, 0.0
    for label, pat in FOCUS_KEYWORD_PATTERNS:
        if pat.search(str(text)):
            return label, 0.6
    return None, 0.0

print('Normalization map:', len(FOCUS_NORMALIZATION), 'entries')
print('Keyword patterns:', len(FOCUS_KEYWORD_PATTERNS), 'entries')

Normalization map: 56 entries
Keyword patterns: 28 entries


In [3]:
focus_raw_list, focus_norm_list, focus_ann_src_list, focus_conf_list = [], [], [], []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Focus annotation'):
    dataset  = row['source_dataset']
    label_raw = row['source_label_raw']
    utterance = row['utterance']

    if dataset == 'MedQuAD':
        if label_raw and not (isinstance(label_raw, float) and np.isnan(label_raw)):
            fr, fn, src, conf = str(label_raw), normalize_focus(str(label_raw)), 'source_metadata', 1.0
        else:
            fr, conf = infer_focus_from_text(utterance)
            fn, src = normalize_focus(fr) if fr else None, 'rule_based'
    else:  # MedDialog
        fr, conf = infer_focus_from_text(utterance)
        fn, src = normalize_focus(fr) if fr else None, 'rule_based'

    focus_raw_list.append(fr)
    focus_norm_list.append(fn)
    focus_ann_src_list.append(src)
    focus_conf_list.append(conf)

df['focus_raw']        = focus_raw_list
df['focus_normalized'] = focus_norm_list
df['_focus_ann_src']   = focus_ann_src_list
df['_focus_conf']      = focus_conf_list

print('Focus annotation complete.')
print('focus_raw non-null:', df['focus_raw'].notna().sum())
print()
print('Top focus_normalized (MedQuAD):')
print(df[df['source_dataset']=='MedQuAD']['focus_normalized'].value_counts().head(10))

Focus annotation:   0%|          | 0/257154 [00:00<?, ?it/s]

Focus annotation complete.
focus_raw non-null: 180817

Top focus_normalized (MedQuAD):
focus_normalized
myocardial_infarction    4072
cancer                   1166
diabetes                  568
breast_cancer             116
hypertension              102
prostate_cancer            96
stroke                     90
thyroid_disorder           86
macular_degeneration       68
lung_cancer                68
Name: count, dtype: int64


## 2. Primary Intent

Inferred from utterance text for both datasets using ordered pattern matching.
Assistant turns always get `other`.

In [4]:
VALID_INTENTS = {
    'information_seeking', 'symptom_inquiry', 'diagnosis_inquiry', 'treatment_inquiry',
    'medication_inquiry', 'test_or_diagnosis', 'prevention', 'risk_factors',
    'cause_or_mechanism', 'prognosis', 'follow_up', 'clarification',
    'emergency_or_urgent', 'other'
}

INTENT_PATTERNS_USER = [
    ('treatment_inquiry',   re.compile(r'\b(treat|treatment|therapy|therapies|cure|manage|management|medication for|medicine for|drug for)\b', re.I)),
    ('medication_inquiry',  re.compile(r'\b(medication|medicine|drug|drugs|pill|prescription|dose|dosage|side effect)\b', re.I)),
    ('symptom_inquiry',     re.compile(r'\b(symptom|sign|feel|feeling|experience|suffer|complaint)\b', re.I)),
    ('diagnosis_inquiry',   re.compile(r'\b(diagnos|test|screening|detect|identify|how do (?:you|doctors?) know)\b', re.I)),
    ('test_or_diagnosis',   re.compile(r'\b(test|exam|lab|laboratory|imaging|scan|x-ray|mri|ct scan|biopsy)\b', re.I)),
    ('prevention',          re.compile(r'\b(prevent|prevention|avoid|reducing risk|lower risk|protective)\b', re.I)),
    ('risk_factors',        re.compile(r'\b(risk factor|risk of|who is at risk|prone to|predispos)\b', re.I)),
    ('cause_or_mechanism',  re.compile(r'\b(cause|causes|why|mechanism|what leads to|how does|origin|etiology)\b', re.I)),
    ('prognosis',           re.compile(r'\b(prognos|outcome|survival|life expectancy|recover|recovery|how long)\b', re.I)),
    ('emergency_or_urgent', re.compile(r'\b(emergency|urgent|immediately|chest pain|can.t breathe|severe|911|call doctor)\b', re.I)),
    ('information_seeking', re.compile(r'\b(what is|what are|tell me|explain|describe|definition|overview|information about)\b', re.I)),
]

def infer_intent(text, speaker):
    if speaker == 'assistant':
        return 'other', 'rule_based', 0.9
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return 'other', 'rule_based', 0.3
    for intent, pat in INTENT_PATTERNS_USER:
        if pat.search(str(text)):
            return intent, 'rule_based', 0.7
    return 'information_seeking', 'rule_based', 0.5

intent_list, intent_ann_src_list, intent_conf_list = [], [], []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Intent annotation'):
    intent, src, conf = infer_intent(row['utterance'], row['speaker'])
    intent_list.append(intent)
    intent_ann_src_list.append(src)
    intent_conf_list.append(conf)

df['primary_intent']  = intent_list
df['_intent_ann_src'] = intent_ann_src_list
df['_intent_conf']    = intent_conf_list

print('Intent distribution:')
print(df['primary_intent'].value_counts())
invalid = df[~df['primary_intent'].isin(VALID_INTENTS)]
print(f'Invalid intent values: {len(invalid)}')
assert len(invalid) == 0

Intent annotation:   0%|          | 0/257154 [00:00<?, ?it/s]

Intent distribution:
primary_intent
other                  128577
information_seeking     63596
symptom_inquiry         19268
medication_inquiry      11117
treatment_inquiry       10868
cause_or_mechanism       6147
diagnosis_inquiry        5540
test_or_diagnosis        4991
emergency_or_urgent      4263
prognosis                1373
prevention                910
risk_factors              504
Name: count, dtype: int64
Invalid intent values: 0


## 3. Dialogue Act

Inferred from speaker, utterance content, and punctuation.

In [5]:
VALID_DIALOGUE_ACTS = {
    'question', 'answer', 'clarification_request', 'clarification',
    'follow_up', 'confirmation', 'correction', 'greeting', 'closing', 'statement', 'other'
}

RE_QUESTION      = re.compile(r'\?\s*$')
RE_QUESTION_WORD = re.compile(r'^\s*(what|how|why|when|where|who|which|can|could|should|is|are|do|does|did|has|have|will|would)\b', re.I)
RE_GREETING      = re.compile(r'^\s*(hello|hi|hey|good morning|good afternoon|dear doctor|dear sir|dear madam)\b', re.I)
RE_CLOSING       = re.compile(r'\b(thank you|thanks|goodbye|bye|regards|best regards|take care)\s*[.!]?\s*$', re.I)
RE_CLARIF_REQ    = re.compile(r'\b(could you clarify|what do you mean|can you explain|please elaborate|what exactly)\b', re.I)
RE_FOLLOW_UP     = re.compile(r'^\s*(also|additionally|furthermore|another question|one more|follow.?up|what about|how about)\b', re.I)
RE_CONFIRMATION  = re.compile(r'^\s*(yes|no|correct|right|exactly|confirmed|that.s right|indeed|absolutely|certainly)\b', re.I)

def infer_dialogue_act(speaker, utterance):
    if utterance is None or (isinstance(utterance, float) and np.isnan(utterance)):
        return ('question' if speaker == 'user' else 'answer'), 'rule_based', 0.5
    text  = str(utterance).strip()
    words = text.split()
    if RE_GREETING.match(text) and len(words) <= 10:
        return 'greeting', 'rule_based', 0.85
    if RE_CLOSING.search(text) and len(words) <= 15:
        return 'closing', 'rule_based', 0.85
    if RE_CLARIF_REQ.search(text):
        return 'clarification_request', 'rule_based', 0.8
    if RE_CONFIRMATION.match(text) and len(text) < 100:
        return 'confirmation', 'rule_based', 0.75
    if RE_FOLLOW_UP.match(text):
        return 'follow_up', 'rule_based', 0.75
    if speaker == 'user':
        if RE_QUESTION.search(text) or RE_QUESTION_WORD.match(text):
            return 'question', 'rule_based', 0.85
        return 'statement', 'rule_based', 0.7
    else:
        if RE_QUESTION.search(text) and len(words) < 30:
            return 'clarification_request', 'rule_based', 0.75
        return 'answer', 'rule_based', 0.85

act_list, act_ann_src_list, act_conf_list = [], [], []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Dialogue act'):
    act, src, conf = infer_dialogue_act(row['speaker'], row['utterance'])
    act_list.append(act)
    act_ann_src_list.append(src)
    act_conf_list.append(conf)

df['dialogue_act']  = act_list
df['_act_ann_src']  = act_ann_src_list
df['_act_conf']     = act_conf_list

print('Dialogue act distribution:')
print(df['dialogue_act'].value_counts())
invalid = df[~df['dialogue_act'].isin(VALID_DIALOGUE_ACTS)]
assert len(invalid) == 0, f'{len(invalid)} invalid dialogue act values'

Dialogue act:   0%|          | 0/257154 [00:00<?, ?it/s]

Dialogue act distribution:
dialogue_act
answer                   128382
statement                 75120
question                  53198
clarification_request       435
greeting                     12
follow_up                     6
closing                       1
Name: count, dtype: int64


## 4. Medical Entity Extraction

In [6]:
MEDICAL_ENTITIES = [
    ('diabetes',              re.compile(r'\bdiabet\w*\b', re.I)),
    ('hypertension',          re.compile(r'\b(hypertension|high blood pressure)\b', re.I)),
    ('myocardial infarction', re.compile(r'\b(heart attack|myocardial infarction)\b', re.I)),
    ('stroke',                re.compile(r'\bstroke\b', re.I)),
    ('asthma',                re.compile(r'\basthma\b', re.I)),
    ('copd',                  re.compile(r'\b(copd|chronic obstructive pulmonary)\b', re.I)),
    ('cancer',                re.compile(r'\b(cancer|carcinoma|neoplasm|tumor|tumour)\b', re.I)),
    ('glaucoma',              re.compile(r'\bglaucoma\b', re.I)),
    ('arthritis',             re.compile(r'\barthritis\b', re.I)),
    ('depression',            re.compile(r'\b(depression|depressive disorder)\b', re.I)),
    ('anxiety',               re.compile(r'\b(anxiety|anxiety disorder)\b', re.I)),
    ('pneumonia',             re.compile(r'\bpneumonia\b', re.I)),
    ('influenza',             re.compile(r'\b(influenza|\bflu\b)\b', re.I)),
    ('covid-19',              re.compile(r'\b(covid.?19|coronavirus|sars-cov-2)\b', re.I)),
    ('ibs',                   re.compile(r'\b(ibs|irritable bowel syndrome)\b', re.I)),
    ('gerd',                  re.compile(r'\b(gerd|acid reflux|gastroesophageal reflux)\b', re.I)),
    ('alzheimers disease',    re.compile(r"\b(alzheimer.?s disease|alzheimers|alzheimer)\b", re.I)),
    ('parkinsons disease',    re.compile(r"\b(parkinson.?s disease|parkinson)\b", re.I)),
    ('migraine',              re.compile(r'\bmigraine\b', re.I)),
    ('hypothyroidism',        re.compile(r'\bhypothyroidism\b', re.I)),
    ('hyperthyroidism',       re.compile(r'\bhyperthyroidism\b', re.I)),
    ('kidney disease',        re.compile(r'\b(kidney disease|renal disease|nephropathy)\b', re.I)),
    ('liver disease',         re.compile(r'\b(liver disease|hepatitis|cirrhosis)\b', re.I)),
    ('dizziness',             re.compile(r'\b(dizziness|dizzy|vertigo)\b', re.I)),
    ('chest pain',            re.compile(r'\bchest pain\b', re.I)),
    ('nausea',                re.compile(r'\bnausea\b', re.I)),
    ('fever',                 re.compile(r'\bfever\b', re.I)),
    ('fatigue',               re.compile(r'\b(fatigue|tired|exhaustion)\b', re.I)),
    ('headache',              re.compile(r'\bheadache\b', re.I)),
    ('back pain',             re.compile(r'\bback pain\b', re.I)),
    ('shortness of breath',   re.compile(r'\b(shortness of breath|dyspnea|breathless)\b', re.I)),
    ('insulin',               re.compile(r'\binsulin\b', re.I)),
    ('metformin',             re.compile(r'\bmetformin\b', re.I)),
    ('aspirin',               re.compile(r'\baspirin\b', re.I)),
    ('ibuprofen',             re.compile(r'\bibuprofen\b', re.I)),
    ('paracetamol',           re.compile(r'\b(paracetamol|acetaminophen)\b', re.I)),
    ('antibiotics',           re.compile(r'\bantibiotic\w*\b', re.I)),
    ('blood glucose',         re.compile(r'\b(blood glucose|blood sugar|hba1c|a1c)\b', re.I)),
    ('blood pressure',        re.compile(r'\bblood pressure\b', re.I)),
    ('cholesterol',           re.compile(r'\bcholesterol\b', re.I)),
    ('surgery',               re.compile(r'\b(surgery|surgical|operation)\b', re.I)),
    ('mri',                   re.compile(r'\bmri\b', re.I)),
    ('x-ray',                 re.compile(r'\bx.?ray\b', re.I)),
]

def extract_entities(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return []
    found, seen = [], set()
    for entity, pat in MEDICAL_ENTITIES:
        if entity not in seen and pat.search(str(text)):
            found.append(entity)
            seen.add(entity)
    return found

df['medical_entities'] = df['utterance'].progress_apply(extract_entities)

has_ent = df['medical_entities'].apply(len) > 0
print(f'Turns with entities: {has_ent.sum():,} / {len(df):,} ({100*has_ent.mean():.1f}%)')
all_ents = [e for lst in df['medical_entities'] for e in lst]
print('\nTop 15 entities:')
print(pd.Series(all_ents).value_counts().head(15))

  0%|          | 0/257154 [00:00<?, ?it/s]

Turns with entities: 120,825 / 257,154 (47.0%)

Top 15 entities:
surgery           18425
antibiotics       17339
cancer            14818
fever             12580
x-ray             12166
mri                9905
anxiety            8194
blood pressure     7868
diabetes           7266
fatigue            5921
dizziness          5239
back pain          4943
hypertension       4852
headache           4301
chest pain         4242
Name: count, dtype: int64


## 5. Annotation Source and Confidence

In [7]:
VALID_ANN_SOURCES = {'source_metadata', 'rule_based', 'derived_from_context', 'constructed', 'model_assisted'}

def combine_source(row):
    sources = [row['_focus_ann_src'], row['_intent_ann_src'], row['_act_ann_src']]
    if 'source_metadata' in sources:
        return 'source_metadata'
    if 'derived_from_context' in sources:
        return 'derived_from_context'
    return 'rule_based'

def combine_conf(row):
    vals = [c for c in [row['_focus_conf'], row['_intent_conf'], row['_act_conf']] if c > 0]
    return round(sum(vals) / len(vals), 3) if vals else None

df['annotation_source']     = df.apply(combine_source, axis=1)
df['annotation_confidence'] = df.apply(combine_conf, axis=1)

print('annotation_source distribution:')
print(df['annotation_source'].value_counts())
print()
print('annotation_confidence stats:')
print(df['annotation_confidence'].describe())

assert df[~df['annotation_source'].isin(VALID_ANN_SOURCES)].empty
assert df[df['annotation_confidence'].notna() & ((df['annotation_confidence']<0)|(df['annotation_confidence']>1))].empty
print('All annotation checks passed.')

annotation_source distribution:
annotation_source
rule_based         224358
source_metadata     32796
Name: count, dtype: int64

annotation_confidence stats:
count    257154.000000
mean          0.756683
std           0.093355
min           0.600000
25%           0.667000
50%           0.783000
75%           0.783000
max           0.917000
Name: annotation_confidence, dtype: float64
All annotation checks passed.


## 6. Save

In [8]:
DROP_COLS = ['_focus_ann_src', '_focus_conf', '_intent_ann_src', '_intent_conf', '_act_ann_src', '_act_conf']
SEMANTIC_COLS = [
    'dialogue_id', 'turn_id', 'speaker', 'utterance',
    'focus_raw', 'focus_normalized', 'primary_intent', 'dialogue_act', 'medical_entities',
    'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin',
    'annotation_source', 'annotation_confidence'
]

semantic_df = df.drop(columns=DROP_COLS)[SEMANTIC_COLS]
out = SEMANTIC_DIR / 'harmonized_semantic.parquet'
semantic_df.to_parquet(out, index=False)

print('Saved:', out, f'({out.stat().st_size/1024/1024:.1f} MB)')
print('Shape:', semantic_df.shape)
print()
print('Intent distribution:')
print(semantic_df['primary_intent'].value_counts())
print()
print('Annotation source distribution:')
print(semantic_df['annotation_source'].value_counts())

Saved: ..\data\processed\semantic\harmonized_semantic.parquet (81.9 MB)
Shape: (257154, 15)

Intent distribution:
primary_intent
other                  128577
information_seeking     63596
symptom_inquiry         19268
medication_inquiry      11117
treatment_inquiry       10868
cause_or_mechanism       6147
diagnosis_inquiry        5540
test_or_diagnosis        4991
emergency_or_urgent      4263
prognosis                1373
prevention                910
risk_factors              504
Name: count, dtype: int64

Annotation source distribution:
annotation_source
rule_based         224358
source_metadata     32796
Name: count, dtype: int64
